In [ ]:
"""
Author: Emily Vosper + Ruby Lieber
Affiliation: University of Bristol
Date: 26/06/2025

Description:
This script plots heat-related mortality attributable to human-induced climate change
across Brazilian state capitals between 2000 and 2019, using factual and counterfactual
simulations from a multi-model ensemble.
"""

In [ ]:
import os
import os
os.environ["PROJ_DATA"] = "/user/home/al18709/breathe/.pixi/envs/default/share/proj"
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
import geopandas as gpd
import seaborn as sns
from dask.distributed import Client, LocalCluster
from dask_jobqueue import SLURMCluster
import pickle
import warnings
warnings.filterwarnings("ignore")

In [ ]:
def prepare_igr_geodataframe(results, geometry_path):
    """
    Merge IGR-level results with geometries for plotting.
    
    results      : the results dict from compute_summary
    geometry_path: path to shapefile/parquet with IGR geometries and 'rgi' column
    """
    geometry_gdf = gpd.read_file(geometry_path)
    geometry_gdf['rgi'] = geometry_gdf['rgi'].astype(str)

    # Pivot the leaf DataFrame from long (statistic as index) to wide (one column per statistic)
    igr_rows = []
    for igr, df in results['igr'].items():
        row = df['mean'].to_dict()
        row['p2.5']  = df['p2.5'].to_dict()
        row['p97.5'] = df['p97.5'].to_dict()
        row['rgi']   = str(igr)
        igr_rows.append(row)
    
    # Cleaner approach — stack into wide format directly
    igr_rows = []
    for igr, df in results['igr'].items():
        row = {'rgi': str(igr)}
        for statistic in df.index:
            row[f'{statistic}_mean']  = df.loc[statistic, 'mean']
            row[f'{statistic}_p2.5']  = df.loc[statistic, 'p2.5']
            row[f'{statistic}_p97.5'] = df.loc[statistic, 'p97.5']
        igr_rows.append(row)

    results_df = pd.DataFrame(igr_rows)
    results_gdf = geometry_gdf.merge(results_df, on='rgi', how='left')

    return results_gdf

In [ ]:
def annotate_capitals(ax, cities, fontsize=18):
    """Scatter and label Brazilian state capitals on a map axis."""
    for city in cities:
        marker = city.get('marker', 'o')
        s = city.get('s', 14)
        ax.scatter(city['x'], city['y'], color='black', s=s, marker=marker,zorder=1000)
        ax.annotate(city['name'], xy=(city['ax'], city['ay']), xycoords='data', fontsize=fontsize)


In [ ]:
def plot_ci_barchart(ax, df_sorted, x_col, x_lower_col, x_upper_col, region_color_map,
                     xlabel, title):
    """Plot a horizontal confidence-interval bar chart sorted by region."""
    for i, row in df_sorted.iterrows():
        color = region_color_map[row["Region"]]
        ax.fill_betweenx(
            [i - 0.2, i + 0.2],
            row[x_lower_col], row[x_upper_col],
            color=color, alpha=0.4
        )
        ax.plot(
            [row[x_col], row[x_col]],
            [i - 0.25, i + 0.25],
            color=color, lw=2
        )

    ax.set_yticks(range(len(df_sorted)))
    ax.set_yticklabels(df_sorted["nome_rgi"], fontsize=18)
    ax.set_xlabel(xlabel, fontsize=20, labelpad=10)
    ax.set_title(title, fontsize=22, pad=10)
    ax.axvline(x=0, color='#4a4949', linewidth=1.5, alpha=0.8, linestyle='--')
    ax.grid(axis='x', linestyle='--', alpha=0.3)
    ax.spines['left'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.set_ticks_position('none')
    ax.tick_params(axis='y', length=0)
    xmin = min(df_sorted[x_lower_col].min(), 0) - 0.5
    xmax = df_sorted[x_upper_col].max() + 0.5
    ax.set_xlim(xmin, xmax)
    ax.set_aspect('auto')
    ax.tick_params(axis='both', which='major', labelsize=18)
    return ax


def add_region_inset(fig, macro, region_color_map, position):
    """Add a small colour-coded region inset map to the figure."""
    ax_inset = fig.add_axes(position)
    for region in macro.index:
        color = region_color_map.get(region, 'lightgrey')
        macro.loc[[region]].plot(ax=ax_inset, color=color, edgecolor='white', alpha=0.4)
    ax_inset.axis('off')
    for region, row in macro.iterrows():
        centroid = row.geometry.centroid
        ax_inset.text(
            centroid.x, centroid.y, region,
            color=region_color_map.get(region, 'black'),
            fontsize=14, weight='bold', ha='center', va='center'
        )
    return ax_inset

In [ ]:
def plot_brazil(ax, fig, results_gdf, statistic='attributable_mortality_per_100k', change_levels=False):
    
    col = f'{statistic}_mean'

    cmap         = plt.get_cmap("plasma_r")
    cropped_cmap = cmap(np.linspace(0.1, 1.0, 256))
    new_cmap     = mcolors.ListedColormap(cropped_cmap)
    new_cmap.set_under('lightgrey')

    if change_levels:
        bounds = change_levels
        norm   = mcolors.BoundaryNorm(bounds, ncolors=256)
    elif 'per_n_person_year' in statistic:
        bounds = [0, 20, 40, 60, 80, 100, 120, 140, 160, 180]
        norm   = mcolors.BoundaryNorm(bounds, ncolors=256, extend='max')
    elif 'fraction' in statistic:
        bounds = [0.00000000001, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5, 5.5, 6, 6.5, 7]
        norm   = mcolors.BoundaryNorm(bounds, ncolors=256, extend='max')
    else:
        bounds = [0, 20, 40, 60, 80, 100, 120, 140, 160, 180]
        norm   = mcolors.BoundaryNorm(bounds, ncolors=256, extend='max')

    results_gdf.plot(
        column=col, ax=ax, cmap=new_cmap, norm=norm,
        vmin=0, vmax=180, legend=False,
        edgecolor="#f5cca4", linewidth=0.2, alpha=0.9
    )

    sm      = plt.cm.ScalarMappable(cmap=new_cmap, norm=norm)
    sm._A   = []
    cbar_ax = fig.add_axes([0.15, 0.05, 0.35, 0.015])
    cbar    = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal', pad=0, ticks=bounds, extend='both')

    if 'per_n_person_year' in statistic:
        cbar.set_label("Attributable mortality (per million person-years) \n", fontsize=20, labelpad=10)
        ax.set_title("Heat-related mortality attributable to human-induced climate change (per million person-years)", fontsize=22, pad=16)
    elif 'heat_related' in statistic:
        cbar.set_label("Attributable mortality (percentage of factual heat-related deaths)", fontsize=20, labelpad=10)
        ax.set_title("Heat-related mortality attributable to human-induced climate change \n (percentage of heat-related deaths)", fontsize=22, pad=16)
    else:
        cbar.set_label("Attributable fraction (percentage of non-accidental deaths)", fontsize=20, labelpad=10)
        ax.set_title("Fraction of heat-related mortality attributable to human-induced climate change \n (percentage of non-accidental deaths)", fontsize=22, pad=16)

    cbar.outline.set_visible(False)
    cbar.ax.tick_params(labelsize=18)

    for spine in ["left", "right", "top", "bottom"]:
        ax.spines[spine].set_visible(False)
    ax.set_yticks([])
    ax.set_xticks([])
    ax.set_xlabel("", fontsize=14)
    ax.set_ylabel("", fontsize=14)

    return ax, fig

In [ ]:
def load_results(path):
    with open(path, 'rb') as f:
        results = pickle.load(f)
    return results

In [ ]:
cities = [
    {'name': 'Rio Branco',     'x': -67.809, 'y':  -9.974, 'ax': -67.4,        'ay': -10.6},
    {'name': 'Maceió',         'x': -35.735, 'y':  -9.666, 'ax': -35.3,        'ay': -10.2},
    {'name': 'Macapá',         'x': -51.050, 'y':   0.034, 'ax': -50.6,        'ay':   0.0},
    {'name': 'Manaus',         'x': -60.021, 'y':  -3.122, 'ax': -59.6,        'ay':  -3.9},
    {'name': 'Salvador',       'x': -38.505, 'y': -12.986, 'ax': -38.2,        'ay': -13.6},
    {'name': 'Fortaleza',      'x': -38.529, 'y':  -3.724, 'ax': -38.2,        'ay':  -3.4},
    {'name': 'Brasília',       'x': -47.895, 'y': -15.807, 'ax': -49.9,        'ay': -15.3, 'marker': 's'},
    {'name': 'Vitória',        'x': -40.308, 'y': -20.315, 'ax': -39.9,        'ay': -21.0},
    {'name': 'Goiânia',        'x': -49.264, 'y': -16.686, 'ax': -48.9,        'ay': -17.4},
    {'name': 'São Luís',       'x': -44.302, 'y':  -2.530, 'ax': -44.0,        'ay':  -2.4},
    {'name': 'Cuiabá',         'x': -56.098, 'y': -15.601, 'ax': -55.7,        'ay': -16.1},
    {'name': 'Campo Grande',   'x': -54.620, 'y': -20.469, 'ax': -59.2,        'ay': -20.7},
    {'name': 'Belo Horizonte', 'x': -43.938, 'y': -19.920, 'ax': -43.5,        'ay': -19.8},
    {'name': 'Belém',          'x': -48.503, 'y':  -1.455, 'ax': -48.0,        'ay':  -2.1},
    {'name': 'João Pessoa',    'x': -34.864, 'y':  -7.121, 'ax': -34.4,        'ay':  -7.35},
    {'name': 'Curitiba',       'x': -49.265, 'y': -25.428, 'ax': -48.8,        'ay': -26.2},
    {'name': 'Recife',         'x': -34.883, 'y':  -8.059, 'ax': -34.5,        'ay':  -8.7},
    {'name': 'Teresina',       'x': -42.803, 'y':  -5.090, 'ax': -45.4,        'ay':  -5.8},
    {'name': 'Rio de Janeiro', 'x': -43.189, 'y': -22.917, 'ax': -42.8,        'ay': -23.7},
    {'name': 'Natal',          'x': -35.209, 'y':  -5.795, 'ax': -34.8,        'ay':  -6.0},
    {'name': 'Porto Alegre',   'x': -51.211, 'y': -30.038, 'ax': -50.8,        'ay': -30.8},
    {'name': 'Porto Velho',    'x': -63.903, 'y':  -8.761, 'ax': -63.5,        'ay':  -9.4},
    {'name': 'Boa Vista',      'x': -60.673, 'y':   2.823, 'ax': -60.2,        'ay':   2.9},
    {'name': 'Florianópolis',  'x': -48.548, 'y': -27.595, 'ax': -48.1,        'ay': -28.3},
    {'name': 'São Paulo',      'x': -46.638, 'y': -23.568, 'ax': -46.3,        'ay': -24.6},
    {'name': 'Aracaju',        'x': -37.073, 'y': -10.947, 'ax': -36.7,        'ay': -11.6},
    {'name': 'Palmas',         'x': -48.333, 'y': -10.184, 'ax': -47.9,        'ay': -10.9},
]

In [ ]:
# ============================================================
# DATA LOADING
# ============================================================

# Load shapefiles
polys = '/bp1/geog-tropical/data/BREATHE/RG2017_rgi_20180911/RG2017_rgi.shp'
gdf = gpd.read_file(polys)

polys_states = '/bp1/geog-tropical/data/BREATHE/shapefiles/BR_UF_2022.shp'
gdf_states = gpd.read_file(polys_states)

# Build macroregion polygons from state boundaries
macroregion_map = {
    'RO': 'North',    'AC': 'North',     'AM': 'North',       'RR': 'North',
    'PA': 'North',    'AP': 'North',     'TO': 'North',
    'MA': 'Northeast','PI': 'Northeast', 'CE': 'Northeast',   'RN': 'Northeast',
    'PB': 'Northeast','PE': 'Northeast', 'AL': 'Northeast',   'SE': 'Northeast', 'BA': 'Northeast',
    'MT': 'Central-West','MS': 'Central-West','GO': 'Central-West','DF': 'Central-West',
    'SP': 'Southeast','RJ': 'Southeast', 'MG': 'Southeast',   'ES': 'Southeast',
    'PR': 'South',    'SC': 'South',     'RS': 'South'
}
gdf_states["macroregion"] = gdf_states["SIGLA_UF"].map(macroregion_map)
macro = gdf_states.dissolve(by="macroregion")

# Load results and merge with geometries
results = load_results('/bp1/geog-tropical/data/BREATHE/results/results_100M.pkl')
results_gdf = prepare_igr_geodataframe(results, geometry_path=polys)

# Derive region from rgi prefix (replaces rgi_region_map lookup)
uf_code_to_region = {
    11: 'North', 12: 'North', 13: 'North', 14: 'North', 15: 'North', 16: 'North', 17: 'North',
    21: 'Northeast', 22: 'Northeast', 23: 'Northeast', 24: 'Northeast', 25: 'Northeast',
    26: 'Northeast', 27: 'Northeast', 28: 'Northeast', 29: 'Northeast',
    31: 'Southeast', 32: 'Southeast', 33: 'Southeast', 35: 'Southeast',
    41: 'South', 42: 'South', 43: 'South',
    50: 'Central-West', 51: 'Central-West', 52: 'Central-West', 53: 'Central-West',
}
results_gdf['Region'] = results_gdf['rgi'].apply(
    lambda x: uf_code_to_region.get(int(str(x)[:2]), None)
)

# Split into present/missing for plotting
df_merged  = results_gdf.dropna(subset=['attributable_mortality_per_100k_mean'])
gdf_missing = results_gdf[results_gdf['attributable_mortality_per_100k_mean'].isna()]

# RGI-to-region lookup for the 27 state capitals
rgi_region_map = {
    '120001': "North",        '270001': "Northeast",   '160001': "North",
    '130001': "North",        '290001': "Northeast",   '230001': "Northeast",
    '530001': "Central-West", '320001': "Southeast",   '520001': "Central-West",
    '210001': "Northeast",    '510001': "Central-West",'500001': "Central-West",
    '310001': "Southeast",    '150001': "North",       '250001': "Northeast",
    '410001': "South",        '260001': "Northeast",   '220001': "Northeast",
    '330001': "Southeast",    '240001': "Northeast",   '430001': "South",
    '110001': "North",        '140001': "North",       '420001': "South",
    '350001': "Southeast",    '280001': "Northeast",   '170001': "North"
}

In [ ]:
# loop through igrs and print out ones where attributable mortality is < 0
# Load population and total deaths, indexed by IGR
population_df = pd.read_parquet('/bp1/geog-tropical/data/BREATHE/igr_population.parquet')
population_df['rgi'] = population_df['rgi'].astype(str)
population_df = population_df.set_index('rgi')

"""
regions that have a lower ci below zero tend to have low populations and low factual and cfactual mortality

"""

for igr in results['igr']:
    if results['igr'][igr].loc['attributable_mortality_per_100k', 'p2.5'] < 0:
        print(f"IGR {igr} has negative attributable mortality: {results['igr'][igr].loc['attributable_mortality_per_100k', 'p2.5']} per 100k")
        print(f"mean actual mortality: {results['igr'][igr].loc['factual_mortality', 'mean']} per 100k")
        print(f"mean cfactual mortality: {results['igr'][igr].loc['counterfactual_mortality', 'mean']} per 100k")
        igr_population = population_df.loc[igr, 'population_count']
        print("igr population:", igr_population)
        

In [ ]:
person_years_df = pd.read_excel('/user/home/al18709/breathe/data_analysis/total_person_years.xlsx')

In [ ]:
# ============================================================
# FIGURE: ATTRIBUTABLE MORTALITY PER 100,000
# ============================================================

# IGR code prefix -> region mapping (first two digits of rgi)
uf_code_to_region = {
    11: 'North', 12: 'North', 13: 'North', 14: 'North', 15: 'North', 16: 'North', 17: 'North',
    21: 'Northeast', 22: 'Northeast', 23: 'Northeast', 24: 'Northeast', 25: 'Northeast',
    26: 'Northeast', 27: 'Northeast', 28: 'Northeast', 29: 'Northeast',
    31: 'Southeast', 32: 'Southeast', 33: 'Southeast', 35: 'Southeast',
    41: 'South', 42: 'South', 43: 'South',
    50: 'Central-West', 51: 'Central-West', 52: 'Central-West', 53: 'Central-West',
}

# Add region column to results_gdf derived from rgi
results_gdf['Region'] = results_gdf['rgi'].apply(
    lambda x: uf_code_to_region.get(int(str(x)[:2]), None)
)
# normalise to 1000 person years
results_gdf['attributable_mortality_per_n_person_year_mean'] = results_gdf['attributable_mortality_per_person_year_mean'] * 1000000
results_gdf['attributable_mortality_per_n_person_year_p2.5'] = results_gdf['attributable_mortality_per_person_year_p2.5'] * 1000000
results_gdf['attributable_mortality_per_n_person_year_p97.5'] = results_gdf['attributable_mortality_per_person_year_p97.5'] * 1000000


# Drop IGRs with no region mapping (e.g. missing geometries)
df_filtered = results_gdf.dropna(subset=['Region', 'attributable_mortality_per_person_year_mean']).copy()

# Sort within regions
region_order = ["South", "Southeast", "Central-West", "Northeast", "North"]
df_filtered['Region'] = pd.Categorical(df_filtered['Region'], categories=region_order, ordered=True)
df_sorted = df_filtered.sort_values(
    ['Region', 'attributable_mortality_per_person_year_mean']
).reset_index(drop=True)

region_palette   = sns.color_palette("tab10", n_colors=5)
# region_color_map = dict(zip(region_order, region_palette))
region_color_map = dict(zip(sorted(df_sorted["Region"].unique()), region_palette))


fig = plt.figure(figsize=(26, 20))
gs  = gridspec.GridSpec(nrows=1, ncols=2, width_ratios=[3, 1], wspace=0)



# Left: Brazil choropleth map
ax_map = fig.add_subplot(gs[0, 0])
ax_map.text(0.1, 1.048, 'a.', transform=ax_map.transAxes, fontsize=28, fontweight='bold', va='top', ha='left')
ax_map, fig = plot_brazil(ax_map, fig, results_gdf, statistic='attributable_mortality_per_n_person_year', change_levels=[0, 20, 40, 60, 80, 100, 120, 140, 160, 180, 200])
gdf_missing.plot(ax=ax_map, color='none', edgecolor="#d2a87d", hatch='////', linewidth=0.1)
macro.plot(ax=ax_map, edgecolor="#ffffff", linewidth=1.5, facecolor="none")
ax_map.set_aspect('auto')
annotate_capitals(ax_map, cities, fontsize=16)

# Filter to state capitals only for the bar chart
capital_rgis = set(rgi_region_map.keys())
# df_capitals = df_sorted[df_sorted['rgi'].isin(capital_rgis)].copy()
df_capitals = df_sorted[df_sorted['rgi'].isin(capital_rgis)].copy().reset_index(drop=True)

# Right: CI bar chart
ax_plot = fig.add_subplot(gs[0, 1])
ax_plot.text(-0.26, 1.048, 'b.', transform=ax_plot.transAxes, fontsize=28, fontweight='bold', va='top', ha='left')
plot_ci_barchart(
#     ax_plot, df_capitals,                          # <-- capitals only
#     x_col='attributable_mortality_per_100k_mean',
#     x_lower_col='attributable_mortality_per_100k_p2.5',
#     x_upper_col='attributable_mortality_per_100k_p97.5',
#     region_color_map=region_color_map,
#     xlabel="Attributable Mortality (per 100,000)",
#     title="Attributable Mortality in Brazilian State Capitals\nwith 95% Confidence Intervals"
# )
    ax_plot, df_capitals,                          # <-- capitals only
    x_col='attributable_mortality_per_n_person_year_mean',
    x_lower_col='attributable_mortality_per_n_person_year_p2.5',
    x_upper_col='attributable_mortality_per_n_person_year_p97.5',
    region_color_map=region_color_map,
    xlabel="Attributable Mortality (per million person-years)",
    title="Attributable Mortality in Brazilian State Capitals\nwith 95% Confidence Intervals"
)

# Region inset map
add_region_inset(fig, macro, region_color_map, position=[0.9, 0.6, 0.18, 0.39])

plt.tight_layout()
plt.savefig('Figure_2.svg', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
fig_a = plt.figure(figsize=(19.5, 20))
ax_map = fig_a.add_subplot(111)
ax_map.text(0.1, 1.048, 'a.', transform=ax_map.transAxes, fontsize=28, fontweight='bold', va='top', ha='left')

ax_map, fig_a = plot_brazil(
    ax_map, fig_a, results_gdf,
    statistic='attributable_mortality_per_n_person_year',
    change_levels=[0, 20, 40, 60, 80, 100, 120, 140, 160, 180, 200]
)
gdf_missing.plot(ax=ax_map, color='none', edgecolor="#d2a87d", hatch='////', linewidth=0.1)
macro.plot(ax=ax_map, edgecolor="#ffffff", linewidth=1.5, facecolor="none")
ax_map.set_aspect('auto')
annotate_capitals(ax_map, cities, fontsize=16)

# --- Center the colorbar ---
# plot_brazil likely adds the colorbar as a separate axis on fig_a.
# Grab it (usually the last axis added) and re-center it horizontally.
cbar_ax = fig_a.axes[-1]  # adjust index if plot_brazil returns/creates it differently
pos = cbar_ax.get_position()
new_x0 = (1 - pos.width) / 2  # centers it across the full figure width
cbar_ax.set_position([new_x0, pos.y0 -0.05, pos.width, pos.height])

plt.tight_layout()
fig_a.savefig('Figure_2a.svg', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
fig_b = plt.figure(figsize=(6.5, 20))
ax_plot = fig_b.add_subplot(111)
ax_plot.text(-0.4, 1.09, 'b.', transform=ax_plot.transAxes, fontsize=28, fontweight='bold', va='top', ha='left')

plot_ci_barchart(
    ax_plot, df_capitals,
    x_col='attributable_mortality_per_n_person_year_mean',
    x_lower_col='attributable_mortality_per_n_person_year_p2.5',
    x_upper_col='attributable_mortality_per_n_person_year_p97.5',
    region_color_map=region_color_map,
    xlabel="Attributable Mortality (per million person-years)",
    title="Attributable Mortality in Brazilian State Capitals\nwith 95% Confidence Intervals"
)

# Region inset map — tripled in size vs. the original [0.9, 0.6, 0.18, 0.39],
# repositioned to stay on-canvas for this narrower figure
add_region_inset(fig_b, macro, region_color_map, position=[0.8, 0.6, 0.65, 0.45])

plt.tight_layout()
fig_b.savefig('Figure_2b.svg', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# SUMMARY STATISTICS
# ============================================================

# Pull country-level stats directly from results
country_stats = results['country']['Brazil']

print('Attributable number of heat-related deaths for the whole of Brazil:')
print(f'  {country_stats.loc['attributable_mortality', 'p2.5']:.2f} – {country_stats.loc['attributable_mortality', 'mean']:.2f} – {country_stats.loc['attributable_mortality', 'p97.5']:.2f}')


print('Attributable number of heat-related deaths for the whole of Brazil per year:')
print(f'  {country_stats.loc['attributable_mortality', 'p2.5']/19:.2f} – {country_stats.loc['attributable_mortality', 'mean']/19:.2f} – {country_stats.loc['attributable_mortality', 'p97.5']/19:.2f}')
print('\n')

af_brazil_mean = country_stats.loc['attributable_fraction', 'mean']
af_brazil_25   = country_stats.loc['attributable_fraction', 'p2.5']
af_brazil_975  = country_stats.loc['attributable_fraction', 'p97.5']

heat_af_mean = country_stats.loc['heat_related_attributable_fraction', 'mean']
heat_af_25   = country_stats.loc['heat_related_attributable_fraction', 'p2.5']
heat_af_975  = country_stats.loc['heat_related_attributable_fraction', 'p97.5']

print('Attributable fraction of heat-related deaths for the whole of Brazil:')
print(f'  {heat_af_25:.2f} – {heat_af_mean:.2f} – {heat_af_975:.2f} %')

print('\nAttributable fraction of all non-accidental deaths for the whole of Brazil:')
print(f'  {af_brazil_25:.2f} – {af_brazil_mean:.2f} – {af_brazil_975:.2f} %')

# Detectible signal: lower CI > 0
detectible     = results_gdf[results_gdf['attributable_mortality_per_100k_p2.5'] > 0]
opposite_signal = (results_gdf['attributable_mortality_per_100k_p97.5'] < 0).sum()
print(f'\nRegions with detectible attributable signal (lower CI > 0): {len(detectible)} out of {len(results_gdf)}')
print(f'Regions with opposite detectible change (upper CI < 0): {opposite_signal}')

print('\nLargest AN:')
idx_max = results_gdf['attributable_mortality_per_n_person_year_mean'].idxmax()
print(results_gdf.loc[idx_max, 'nome_rgi'], f"— {results_gdf.loc[idx_max, 'attributable_mortality_per_n_person_year_mean']:.2f} per n_person_years")
print(results_gdf.loc[idx_max, 'nome_rgi'], f"— {results_gdf.loc[idx_max, 'attributable_fraction_mean']:.2f} %")

print('Smallest AN:')
idx_min = results_gdf['attributable_mortality_per_n_person_year_mean'].idxmin()
print(results_gdf.loc[idx_min, 'nome_rgi'], f"— {results_gdf.loc[idx_min, 'attributable_mortality_per_n_person_year_mean']:.2f} per n_person_years")
print(results_gdf.loc[idx_min, 'nome_rgi'], f"— {results_gdf.loc[idx_min, 'attributable_fraction_mean']:.2f} %")

print('\nLargest AF:')
idx_max = results_gdf['attributable_fraction_mean'].idxmax()
print(results_gdf.loc[idx_max, 'nome_rgi'], f"— {results_gdf.loc[idx_max, 'attributable_fraction_mean']:.2f} %")

print('Smallest AF:')
idx_min = results_gdf['attributable_fraction_mean'].idxmin()
print(results_gdf.loc[idx_min, 'nome_rgi'], f"— {results_gdf.loc[idx_min, 'attributable_fraction_mean']:.2f} %")

In [ ]:
person_years_df = pd.read_excel('/user/home/al18709/breathe/data_analysis/total_person_years.xlsx')
person_years_df['rgi'] = person_years_df['cod_rgi'].astype(float).astype('Int64').astype(str)
person_years_df = person_years_df.set_index('rgi')

In [ ]:
barretos_factual_mortality = results_gdf.loc[idx_max, 'factual_mortality_mean']
barretos_person_years = person_years_df.loc['350032', 'total_person_years']

n_factual_per_million_py = (barretos_factual_mortality / barretos_person_years) * 1_000_000
print('n (factual deaths per million person-years):', n_factual_per_million_py)
print('attributable per million person-years:', results_gdf.loc[idx_max, 'attributable_mortality_per_n_person_year_mean'])
print('check %:', results_gdf.loc[idx_max, 'attributable_mortality_per_n_person_year_mean'] / n_factual_per_million_py * 100)
print('heat_related_attributable_fraction_mean:', results_gdf.loc[idx_max, 'heat_related_attributable_fraction_mean'])

In [ ]:
descriptives = pd.read_excel('/user/home/al18709/breathe/data_analysis/Results_COHORT_noRH_tas_ACCESS-CM2_r1i1p1f1.xlsx')
descriptives.rename(columns={'COD_RGI': 'rgi'}, inplace=True)
descriptives['rgi'] = descriptives['rgi'].astype(str)
results_gdf_2 = pd.merge(
    results_gdf, descriptives[['rgi', 'total_obitos']], on='rgi', how='left'
)

In [ ]:
idx = idx_min
barretos_factual_mortality = results_gdf_2.loc[idx, 'total_obitos']
barretos_person_years = person_years_df.loc[results_gdf.loc[idx]['rgi'], 'total_person_years']

n_factual_per_million_py = (barretos_factual_mortality / barretos_person_years) * 1_000_000
print('n (factual deaths per million person-years):', n_factual_per_million_py)
print('attributable per million person-years:', results_gdf.loc[idx, 'attributable_mortality_per_n_person_year_mean'])
print('check %:', results_gdf.loc[idx, 'attributable_mortality_per_n_person_year_mean'] / n_factual_per_million_py * 100)
print('heat_related_attributable_fraction_mean:', results_gdf.loc[idx, 'heat_related_attributable_fraction_mean'])